# WSJ 2027 - Deltagare, IST & Ledare på hemadresser

KeplerGL-karta som visar alla deltagare, IST och ledare plottade på deras **riktiga hemadresser** (postort/postnummer),
uppdelade i 5 lager:
- **Deltagare rundresa** (Scout Blue)
- **Deltagare direktresa** (Ljusblå)
- **IST rundresa** (Guld)
- **IST egen resa** (Ljusorange)
- **Ledare** (Grön)

**Datakälla:** Scoutnet CSV-export (deltagare/IST) + candidates CSV (ledare)

In [1]:
# Cell 1: Load and filter participant data
import pandas as pd
import json
import os
import random
import time
from datetime import date

CSV_FILE = '/config/notebooks/wsj27/participants_20260318.txt'
CACHE_FILE = '/config/notebooks/wsj27/adress_geocode_cache.json'
OUTPUT_DIR = '/config/notebooks/wsj27'

df_raw = pd.read_csv(CSV_FILE, encoding='utf-8')
print(f'Loaded {len(df_raw)} rows')

# Filter: active participants only
df = df_raw[
    (df_raw['Anmäld'] == 'Ja') &
    (df_raw['Återkallelsedatum (om något)'] == '-')
].copy()
print(f'Active (anmäld, ej återkallad): {len(df)}')

# Map fee type to category and travel
FEE_MAP = {
    'Deltagare rundresa ansökningsavgift': ('deltagare', 'rundresa'),
    'Deltagare direktresa ansökningsavgift': ('deltagare', 'direktresa'),
    'IST rundresa ansökningsavgift': ('ist', 'rundresa'),
    'IST egen resa  ansökningsavgift': ('ist', 'egen_resa'),
}

df['category'] = df['Arrangemangsavgift'].map(lambda x: FEE_MAP.get(x, (None, None))[0])
df['travel'] = df['Arrangemangsavgift'].map(lambda x: FEE_MAP.get(x, (None, None))[1])

# Drop CMT/Anställd/unknown
df = df[df['category'].notna()].copy()
print(f'Deltagare + IST: {len(df)}')
print()
print('By category and travel:')
print(df.groupby(['category', 'travel']).size().to_string())

Loaded 2430 rows
Active (anmäld, ej återkallad): 2374
Deltagare + IST: 2352

By category and travel:
category   travel    
deltagare  direktresa     561
           rundresa      1345
ist        egen_resa      302
           rundresa       144


In [15]:
# Cell 2: Geocode unique postal codes
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

# Load cache
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'r', encoding='utf-8') as f:
        geocode_cache = json.load(f)
    print(f'Loaded {len(geocode_cache)} cached entries')
else:
    geocode_cache = {}
    print('No cache, starting fresh')

# Build unique postal code + city + country combos
df['_geo_key'] = (df['Postnummer'].astype(str).str.strip() + '|' +
                  df['Postort'].astype(str).str.strip() + '|' +
                  df['Land'].astype(str).str.strip())
unique_keys = df['_geo_key'].unique()
print(f'Unique postal locations: {len(unique_keys)}')

# Geocode missing
geolocator = Nominatim(user_agent='wsj2027_address_map')
new_count = 0
failed = []

for i, key in enumerate(unique_keys):
    if key in geocode_cache:
        continue

    postnr, ort, land = key.split('|', 2)
    country = land if land and land != 'Sverige' else 'Sweden'
    query = f'{postnr} {ort}, {country}'

    try:
        location = geolocator.geocode(query, timeout=10)
        if location:
            geocode_cache[key] = {'lat': location.latitude, 'lng': location.longitude}
            new_count += 1
        else:
            # Fallback: try just the city + country
            location = geolocator.geocode(f'{ort}, {country}', timeout=10)
            time.sleep(1.1)
            if location:
                geocode_cache[key] = {'lat': location.latitude, 'lng': location.longitude}
                new_count += 1
            else:
                geocode_cache[key] = {'lat': None, 'lng': None}
                failed.append(f'{postnr} {ort} ({land})')
    except (GeocoderTimedOut, Exception) as e:
        geocode_cache[key] = {'lat': None, 'lng': None}
        failed.append(f'{postnr} {ort} ({e})')

    time.sleep(1.1)

    # Save periodically
    if new_count > 0 and new_count % 50 == 0:
        with open(CACHE_FILE, 'w', encoding='utf-8') as f:
            json.dump(geocode_cache, f, ensure_ascii=False, indent=2)
        print(f'  ... geocoded {new_count} so far ({i+1}/{len(unique_keys)})')

# Final save
with open(CACHE_FILE, 'w', encoding='utf-8') as f:
    json.dump(geocode_cache, f, ensure_ascii=False, indent=2)

print(f'\nGeocoded {new_count} new locations')
print(f'Total cached: {len(geocode_cache)}')
if failed:
    print(f'Failed ({len(failed)}):')
    for f_name in failed[:20]:
        print(f'  - {f_name}')

Loaded 1579 cached entries
Unique postal locations: 1579

Geocoded 0 new locations
Total cached: 1579


In [ ]:
# Cell 2b: Load and geocode leaders from candidates CSV
import re

LEDARE_CSV = '/config/notebooks/wsj27/candidates (1).csv'
LEDARE_CACHE_FILE = '/config/notebooks/wsj27/ledare_geocode_cache.json'

df_led = pd.read_csv(LEDARE_CSV, encoding='utf-8')
print(f'Loaded {len(df_led)} leaders')

# Clean bostadsort - strip prefix and take first city
df_led['bostadsort'] = df_led['Vilken är din bostadsort?'].str.replace(
    r'^WSJ 2027 - Avdelningsledare - ', '', regex=True).str.strip()
df_led['kar'] = df_led['Vilken scoutkår tillhör du?'].str.replace(
    r'^WSJ 2027 - Avdelningsledare - ', '', regex=True).str.strip()
df_led['namn'] = df_led['Förnamn'] + ' ' + df_led['Efternamn']

def clean_ort(s):
    if not isinstance(s, str):
        return ''
    s = re.split(r',\s*men\s', s)[0]
    parts = [p.strip() for p in s.split(',')]
    return parts[0]

df_led['ort_clean'] = df_led['bostadsort'].apply(clean_ort)

# Load/build geocode cache
if os.path.exists(LEDARE_CACHE_FILE):
    with open(LEDARE_CACHE_FILE, 'r', encoding='utf-8') as f:
        ledare_cache = json.load(f)
    print(f'Loaded {len(ledare_cache)} cached entries')
else:
    ledare_cache = {}

unique_orts = df_led['ort_clean'].unique()
to_geocode_led = [o for o in unique_orts if o and o not in ledare_cache]
print(f'Unique cities: {len(unique_orts)}, need geocoding: {len(to_geocode_led)}')

if to_geocode_led:
    new_count = 0
    for ort in to_geocode_led:
        try:
            location = geolocator.geocode(f'{ort}, Sweden', timeout=10)
            if location:
                ledare_cache[ort] = {'lat': location.latitude, 'lng': location.longitude}
                new_count += 1
            else:
                ledare_cache[ort] = {'lat': None, 'lng': None}
                print(f'  FAILED: {ort}')
        except Exception as e:
            ledare_cache[ort] = {'lat': None, 'lng': None}
            print(f'  ERROR: {ort} ({e})')
        time.sleep(2)
    with open(LEDARE_CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump(ledare_cache, f, ensure_ascii=False, indent=2)
    print(f'Geocoded {new_count} new cities')

ok = sum(1 for v in ledare_cache.values() if v.get('lat') is not None)
print(f'Total OK: {ok}/{len(ledare_cache)}')

In [ ]:
# Cell 3: Assign coordinates with jitter
random.seed(42)
JITTER = 0.003  # ~300m

def get_coords(geo_key):
    """Get lat/lng for a geo key, with random jitter."""
    entry = geocode_cache.get(geo_key, {})
    lat, lng = entry.get('lat'), entry.get('lng')
    if lat is not None and lng is not None:
        return (
            lat + random.uniform(-JITTER, JITTER),
            lng + random.uniform(-JITTER, JITTER)
        )
    return (None, None)

coords = df['_geo_key'].apply(get_coords)
df['lat'] = coords.apply(lambda x: x[0])
df['lng'] = coords.apply(lambda x: x[1])

df_map = df[df['lat'].notna()].copy()
print(f'Deltagare/IST with coordinates: {len(df_map)} / {len(df)} ({len(df_map)/len(df)*100:.1f}%)')

df_export = df_map[['Namn', 'Postort', 'Postnummer', 'Kår', 'category', 'travel', 'lat', 'lng']].copy()
df_export.columns = ['namn', 'postort', 'postnummer', 'kar', 'category', 'travel', 'lat', 'lng']

# Assign coords to leaders
random.seed(99)
def get_ledare_coords(ort):
    entry = ledare_cache.get(ort, {})
    lat, lng = entry.get('lat'), entry.get('lng')
    if lat is not None and lng is not None:
        return (lat + random.uniform(-JITTER, JITTER), lng + random.uniform(-JITTER, JITTER))
    return (None, None)

lcoords = df_led['ort_clean'].apply(get_ledare_coords)
df_led['lat'] = lcoords.apply(lambda x: x[0])
df_led['lng'] = lcoords.apply(lambda x: x[1])
df_led_map = df_led[df_led['lat'].notna()].copy()
print(f'Ledare with coordinates: {len(df_led_map)} / {len(df_led)}')

df_led_export = df_led_map[['namn', 'ort_clean', 'kar', 'lat', 'lng']].copy()
df_led_export.columns = ['namn', 'postort', 'kar', 'lat', 'lng']
df_led_export['category'] = 'ledare'
df_led_export['travel'] = ''
df_led_export['postnummer'] = ''
df_led_export = df_led_export[['namn', 'postort', 'postnummer', 'kar', 'category', 'travel', 'lat', 'lng']]

print(f'\n=== Statistics ===')
for (cat, trv), grp in df_export.groupby(['category', 'travel']):
    print(f'  {cat} {trv}: {len(grp)}')
print(f'  ledare: {len(df_led_export)}')
print(f'  Total: {len(df_export) + len(df_led_export)}')

In [ ]:
# Cell 4: Generate KeplerGL map
try:
    from scoutnet_secrets import MAPBOX_TOKEN
except ImportError:
    MAPBOX_TOKEN = os.environ.get('MAPBOX_TOKEN', '')

# Split into 5 datasets
datasets = {
    'del_rund': df_export[df_export['category'].eq('deltagare') & df_export['travel'].eq('rundresa')],
    'del_direkt': df_export[df_export['category'].eq('deltagare') & df_export['travel'].eq('direktresa')],
    'ist_rund': df_export[df_export['category'].eq('ist') & df_export['travel'].eq('rundresa')],
    'ist_egen': df_export[df_export['category'].eq('ist') & df_export['travel'].eq('egen_resa')],
    'ledare': df_led_export,
}

LAYER_CONFIG = [
    ('del_rund', 'Deltagare rundresa', [0, 75, 135], 6),
    ('del_direkt', 'Deltagare direktresa', [91, 164, 207], 6),
    ('ist_rund', 'IST rundresa', [232, 168, 56], 6),
    ('ist_egen', 'IST egen resa', [240, 208, 128], 6),
    ('ledare', 'Ledare', [76, 175, 80], 8),
]

layers = []
for data_id, label, color, radius in LAYER_CONFIG:
    layers.append({
        'id': data_id,
        'type': 'point',
        'config': {
            'dataId': data_id,
            'label': label,
            'isVisible': True,
            'columns': {'lat': 'lat', 'lng': 'lng'},
            'color': color,
            'visConfig': {
                'radius': radius,
                'fixedRadius': False,
                'opacity': 0.7,
                'outline': False,
                'filled': True,
                'radiusRange': [2, 8],
            }
        }
    })

tooltip_fields = {}
for data_id, _, _, _ in LAYER_CONFIG:
    tooltip_fields[data_id] = [
        {'name': 'namn', 'format': None},
        {'name': 'postort', 'format': None},
        {'name': 'kar', 'format': None},
        {'name': 'category', 'format': None},
        {'name': 'travel', 'format': None},
    ]

kepler_config = {
    'version': 'v1',
    'config': {
        'mapState': {'latitude': 59.0, 'longitude': 16.0, 'zoom': 5},
        'visState': {
            'layers': layers,
            'interactionConfig': {
                'tooltip': {'fieldsToShow': tooltip_fields, 'enabled': True}
            },
        },
    },
}

data_payload = {}
for data_id, _, _, _ in LAYER_CONFIG:
    data_payload[data_id] = datasets[data_id].to_dict(orient='split')

data_config = {
    'config': kepler_config,
    'data': data_payload,
    'options': {'readOnly': False, 'centerMap': False},
}

for data_id, label, _, _ in LAYER_CONFIG:
    print(f'{label}: {len(datasets[data_id])}')
print(f'Total: {sum(len(d) for d in datasets.values())}')

In [ ]:
# Cell 5: Export KeplerGL HTML

HTML_TEMPLATE = '''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>WSJ 2027 - Deltagare, IST & Ledare hemadresser</title>
<link href="https://d1a3f4spazzrp4.cloudfront.net/kepler.gl/uber-fonts/4.0.0/superfine.css" rel="stylesheet">
<link href="https://api.tiles.mapbox.com/mapbox-gl-js/v1.1.1/mapbox-gl.css" rel="stylesheet">
<script src="https://unpkg.com/react@17.0.2/umd/react.production.min.js" crossorigin></script>
<script src="https://unpkg.com/react-dom@17.0.2/umd/react-dom.production.min.js" crossorigin></script>
<script src="https://unpkg.com/redux@3.7.2/dist/redux.js" crossorigin></script>
<script src="https://unpkg.com/react-redux@7.1.3/dist/react-redux.min.js" crossorigin></script>
<script src="https://unpkg.com/react-intl@3.12.0/dist/react-intl.min.js" crossorigin></script>
<script src="https://unpkg.com/react-copy-to-clipboard@5.0.2/build/react-copy-to-clipboard.min.js" crossorigin></script>
<script src="https://unpkg.com/styled-components@4.1.3/dist/styled-components.min.js" crossorigin></script>
<script src="https://unpkg.com/kepler.gl@2.5.5/umd/keplergl.min.js" crossorigin></script>
<style>
  font-family: ff-clan-web-pro, 'Helvetica Neue', Helvetica, sans-serif;
  font-weight: 400; font-size: 0.875em; line-height: 1.71429;
  *, *:before, *:after { box-sizing: border-box; }
  body { margin: 0; padding: 0; }
  #app { width: 100vw; height: 100vh; position: absolute; top: 0; left: 0; }
</style>
</head>
<body>
<div id="app"></div>
<script>window.__keplerglDataConfig = __DATA_CONFIG__;</script>
<script>
(function() {
  var K = window.KeplerGl;
  var Redux = window.Redux;
  var reducer = K.keplerGlReducer.initialState({
    uiState: { currentModal: null, activeSidePanel: null }
  });
  var middlewares = K.enhanceReduxMiddleware([]);
  var store = Redux.createStore(
    Redux.combineReducers({ keplerGl: reducer }),
    {},
    Redux.applyMiddleware.apply(null, middlewares)
  );
  var MapApp = K.injectComponents([]);
  var dims = { w: window.innerWidth, h: window.innerHeight };
  function App() {
    var ref = React.useState(dims);
    var size = ref[0], setSize = ref[1];
    React.useEffect(function() {
      function onResize() { setSize({ w: window.innerWidth, h: window.innerHeight }); }
      window.addEventListener('resize', onResize);
      return function() { window.removeEventListener('resize', onResize); };
    }, []);
    return React.createElement(
      ReactRedux.Provider, { store: store },
      React.createElement(MapApp, {
        mapboxApiAccessToken: '__MAPBOX_TOKEN__',
        id: 'map',
        width: size.w,
        height: size.h,
        appName: 'WSJ 2027 Adresser'
      })
    );
  }
  ReactDOM.render(React.createElement(App), document.getElementById('app'));
  var cfg = window.__keplerglDataConfig;
  var dsets = Object.keys(cfg.data).map(function(key) {
    var d = cfg.data[key];
    return {
      info: { id: key, label: key },
      data: {
        fields: d.columns.map(function(c) { return { name: c }; }),
        rows: d.data
      }
    };
  });
  store.dispatch(K.addDataToMap({
    datasets: dsets,
    config: cfg.config,
    options: cfg.options || { centerMap: true }
  }));
})();
</script>
</body>
</html>'''

data_config_json = json.dumps(data_config)
html_content = (HTML_TEMPLATE
    .replace('__DATA_CONFIG__', data_config_json)
    .replace('__MAPBOX_TOKEN__', MAPBOX_TOKEN))

output_file = f'{OUTPUT_DIR}/wsj_adresser_karta.html'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f'Saved: {output_file}')
print(f'File size: {os.path.getsize(output_file) / 1024 / 1024:.1f} MB')
print()
print('=== Final Statistics ===')
print(f'  Deltagare rundresa:    {len(datasets["del_rund"]):5d}')
print(f'  Deltagare direktresa:  {len(datasets["del_direkt"]):5d}')
print(f'  IST rundresa:          {len(datasets["ist_rund"]):5d}')
print(f'  IST egen resa:         {len(datasets["ist_egen"]):5d}')
print(f'  Ledare:                {len(datasets["ledare"]):5d}')
print(f'  Total:                 {sum(len(d) for d in datasets.values()):5d}')